In [1]:
import regex
from collections import defaultdict
from typing import Iterable, Iterator, List, Set, Tuple
import json 
import regex


In [ ]:
#def from_files(cls, vocab_filepath, merges_filepath, special_tokens=None) 
vocab_filepath = '/root/workspace/gaoguoji-cs336-assignment1/tests/fixtures/train-bpe-reference-vocab.json'
merges_filepath = '/root/workspace/gaoguoji-cs336-assignment1/tests/fixtures/train-bpe-reference-merges.txt'
with open(vocab_filepath) as vocab_f:
    vocab_f = json.load(vocab_f)
vocab = {}
for k, v in vocab_f.items():
    vocab[v] = k.encode('utf-8')
print(f"gpt2_vocab is {vocab}")
merges = []
with open(merges_filepath, 'r', 'utf-8') as f:
    for line in f:
        cleaned_line = line.rstrip()
        if cleaned_line and len(cleaned_line.split(" ")) == 2:
            merges.append((cleaned_line.split(" ")[0].encode('utf-8'), cleaned_line.split(" ")[1].encode('utf-8')))
print(f"gpt2_merges is {merges}")
special_tokens = ["<|endoftext|>"]

gpt2_vocab is {0: b'<|endoftext|>', 1: b'!', 2: b'"', 3: b'#', 4: b'$', 5: b'%', 6: b'&', 7: b"'", 8: b'(', 9: b')', 10: b'*', 11: b'+', 12: b',', 13: b'-', 14: b'.', 15: b'/', 16: b'0', 17: b'1', 18: b'2', 19: b'3', 20: b'4', 21: b'5', 22: b'6', 23: b'7', 24: b'8', 25: b'9', 26: b':', 27: b';', 28: b'<', 29: b'=', 30: b'>', 31: b'?', 32: b'@', 33: b'A', 34: b'B', 35: b'C', 36: b'D', 37: b'E', 38: b'F', 39: b'G', 40: b'H', 41: b'I', 42: b'J', 43: b'K', 44: b'L', 45: b'M', 46: b'N', 47: b'O', 48: b'P', 49: b'Q', 50: b'R', 51: b'S', 52: b'T', 53: b'U', 54: b'V', 55: b'W', 56: b'X', 57: b'Y', 58: b'Z', 59: b'[', 60: b'\\', 61: b']', 62: b'^', 63: b'_', 64: b'`', 65: b'a', 66: b'b', 67: b'c', 68: b'd', 69: b'e', 70: b'f', 71: b'g', 72: b'h', 73: b'i', 74: b'j', 75: b'k', 76: b'l', 77: b'm', 78: b'n', 79: b'o', 80: b'p', 81: b'q', 82: b'r', 83: b's', 84: b't', 85: b'u', 86: b'v', 87: b'w', 88: b'x', 89: b'y', 90: b'z', 91: b'{', 92: b'|', 93: b'}', 94: b'~', 95: b'\xc2\xa1', 96: b'\xc2\xa2'

In [4]:
text = """u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.
<|endoftext|>
Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.
Tom asked his friend, Sam, to help him search for the ball. They looked high and low, but they could not find the ball. Tom said, "I think my ball fell into the pit."
Sam and Tom went close to the pit. They were scared, but they wanted to find the red ball. They looked into the pit, but it was too dark to see. Tom said, "We must go in and search for my ball."
They went into the pit to search. It was dark and scary. They could not find the ball. They tried to get out, but the pit was too deep. Tom and Sam were stuck in the pit. They called for help, but no one could hear them. They were sad and scared, and they never got out of the pit.
<|endoftext|>
Once upon a time, there was a messy giant named Bob. Bob lived in a big forest with many trees. Bob was always making a mess because he was so big and clumsy. One day, Bob decided he wanted to sell his big rocks to the people in the town.
Bob walked to the town with a big bag of rocks. He met a little girl named Sue. Sue looked at the rocks and said, "These rocks are very messy, but I can help you clean them." Bob was happy and thanked Sue. They cleaned the rocks together and made them shiny.
After cleaning the rocks, Bob an"""
paragraphs = regex.split(f'({'|'.join(map(regex.escape, special_tokens))})', text)
[vocab_f[special_token] for special_token in special_tokens]
print(paragraphs)

['u don\'t have to be scared of the loud dog, I\'ll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.\n', '<|endoftext|>', '\nOnce upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.\nTom asked his friend, Sam, to help him search for the ball. They looked high and low, but they could not find the ball. Tom said, "I think my ball fell into the pit."\nSam and Tom went close to the pit. They were scared, but they wanted to find the red ball. They looked into the pit, but it was too dark to see. Tom said, "We must go in and search for my ball."\nThey went into the pit to search. It was dark and scary. They could not find the ball. They tried to get out, but the pit was too deep. Tom and Sam were stuck in the pit. They called for help, but no

In [13]:
result = []
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
merges_priority = {k: i for i, k in enumerate(merges)}
print(f"merges_priority is {merges_priority}")
vocab_bytes = {v: k for k, v in vocab.items()}
print(f"vocab_bytes is {vocab_bytes}")
def encode_text(word_bytes: bytes
    ):
    word_seg = [bytes([ch]) for ch in word_bytes]
    while len(word_seg) > 1:
        #print(f"word seg is {word_seg}")
        merge_candidates = set()
        for i in range(len(word_seg) - 1):
            if (word_seg[i], word_seg[i + 1]) in merges:
                merge_candidates.add((word_seg[i], word_seg[i + 1]))
        if not merge_candidates:
            break 
        best_merge = min(merge_candidates, key = lambda x : merges_priority[x])
        new_seg = []
        #print(f"best merge is {best_merge}")
        i = 0
        while i < len(word_seg):
            if i < len(word_seg) - 1 and ((word_seg[i], word_seg[i + 1]) == best_merge):
                new_seg.append(word_seg[i] + word_seg[i + 1])
                i += 2
            else :
                new_seg.append(word_seg[i])
                i += 1
        word_seg = new_seg 
    #print(f"new seg is {word_seg}")
    return [vocab_bytes[seg] for seg in word_seg]

merges_priority is {(b'\xc4\xa0', b't'): 0, (b'\xc4\xa0', b'a'): 1, (b'h', b'e'): 2, (b'i', b'n'): 3, (b'r', b'e'): 4, (b'o', b'n'): 5, (b'\xc4\xa0t', b'he'): 6, (b'e', b'r'): 7, (b'\xc4\xa0', b's'): 8, (b'a', b't'): 9, (b'\xc4\xa0', b'w'): 10, (b'\xc4\xa0', b'o'): 11, (b'e', b'n'): 12, (b'\xc4\xa0', b'c'): 13, (b'i', b't'): 14, (b'i', b's'): 15, (b'a', b'n'): 16, (b'o', b'r'): 17, (b'e', b's'): 18, (b'\xc4\xa0', b'b'): 19, (b'e', b'd'): 20, (b'\xc4\xa0', b'f'): 21, (b'in', b'g'): 22, (b'\xc4\xa0', b'p'): 23, (b'o', b'u'): 24, (b'\xc4\xa0a', b'n'): 25, (b'a', b'l'): 26, (b'a', b'r'): 27, (b'\xc4\xa0t', b'o'): 28, (b'\xc4\xa0', b'm'): 29, (b'\xc4\xa0o', b'f'): 30, (b'\xc4\xa0', b'in'): 31, (b'\xc4\xa0', b'd'): 32, (b'\xc4\xa0', b'h'): 33, (b'\xc4\xa0an', b'd'): 34, (b'i', b'c'): 35, (b'a', b's'): 36, (b'l', b'e'): 37, (b'\xc4\xa0t', b'h'): 38, (b'i', b'on'): 39, (b'o', b'm'): 40, (b'l', b'l'): 41, (b'en', b't'): 42, (b'\xc4\xa0', b'n'): 43, (b'\xc4\xa0', b'l'): 44, (b's', b't'): 45, (b'

In [14]:
for paragraph in paragraphs:
    if paragraph in special_tokens:
        result.append(vocab_f[paragraph])
    else :
        words = regex.findall(PAT, paragraph)
        word_bytes_list = [word.encode('utf-8') for word in words]
        for word_bytes in word_bytes_list:
            result.extend(encode_text(word_bytes)) 
print(result)

KeyError: b' '